# 🐖 19 | Mapas de calor de rendimiento, conteo y confiabilidad geométrica

**RECURSO: CPU — sin acelerador. No entrena ni repite predicciones.**

**Fuentes de resultados congeladas:** notebooks 09 (COCOeval por fecha), 11 (IoU de parejas emparejadas con umbral fijo), 15 (conteo con umbral seleccionado en validación) y 18 corregido (umbrales de tamaño relativo calculados con validación).

Figuras exportadas en inglés, 600 dpi (PNG), PDF vectorial y SVG, con tamaños de muestra por celda. La congestión es **un proxy de superposición de cajas**, NO una etiqueta de oclusión. El IoU de parejas emparejadas **no es mAP** y excluye cerdos sin correspondencia. Las cinco fechas de prueba no equivalen a cinco granjas independientes.

No se utilizarán las figuras originales del Notebook 18 con terciles defectuosos.

## A. Dependencias y rutas

In [ ]:
%pip -q install "pandas>=2,<3" "openpyxl>=3.1" "matplotlib>=3.7" "numpy>=1.24"


In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, hashlib, zipfile, platform
from datetime import datetime,timezone
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
MET=ROOT/'03_Resultados'/'03_Metricas'; FIG=ROOT/'03_Resultados'/'04_Figuras'
REP=ROOT/'03_Resultados'/'05_Reportes'; EXP=ROOT/'05_Exportaciones_ZIP'
F09=MET/'CERDOS_09_Comparacion_Armonizada_COCO.xlsx'
F11=MET/'CERDOS_11_Confiabilidad_Geometrica.xlsx'
F15=MET/'CERDOS_15_Correccion_Conteo_Val.xlsx'
F18=MET/'CERDOS_18_Tamano_Bootstrap_Sesion.xlsx'
GT=REP/'CERDOS_09_COCO_Armonizado'/'CERDOS_09_GT_test_coco.json'
for p in [F09,F11,F15,F18,GT]:
    if not p.is_file():raise FileNotFoundError('Falta fuente original: '+str(p))
for p in [MET,FIG,REP,EXP]:p.mkdir(parents=True,exist_ok=True)

def sha256(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
SOURCE_HASHES={p.name:sha256(p) for p in [F09,F11,F15,F18,GT]}
print('✅ Cinco fuentes localizadas y verificadas por SHA256')


## B. Lectura y controles de coherencia

In [ ]:
def model_name(x):
    s=str(x)
    return 'YOLO11n-seg' if s.startswith('YOLO') else ('Mask R-CNN' if s.startswith('Mask R-CNN') else s)

ap=pd.read_excel(F09,sheet_name='Por_Fecha_COCO')
pairs=pd.read_excel(F11,sheet_name='Pares_Geometricos')
count=pd.read_excel(F15,sheet_name='Por_imagen')
size_method=pd.read_excel(F18,sheet_name='Metodo_tamanos').iloc[0]
images={int(x['id']):x for x in json.loads(GT.read_text(encoding='utf8'))['images']}
assert len(images)==373 and len(ap)==10 and len(pairs)==9069
assert 0 < float(size_method['q33_val']) < float(size_method['q67_val']) <1
q1,q2=float(size_method['q33_val']),float(size_method['q67_val'])
for df in [ap,pairs,count]:df['modelo']=df['modelo'].map(model_name)
assert set(ap['modelo'])==set(pairs['modelo'])=={'YOLO11n-seg','Mask R-CNN'}
assert ap.groupby('modelo')['fecha'].nunique().eq(5).all()
assert pairs['image_id'].isin(images).all()
area=pd.Series({i:int(d['width'])*int(d['height']) for i,d in images.items()})
pairs['area_rel']=pairs['gt_area_px2'].to_numpy(dtype=float)/pairs['image_id'].map(area).to_numpy(dtype=float)
pairs['tercil']=pd.cut(pairs['area_rel'],bins=[-np.inf,q1,q2,np.inf],labels=['Small','Medium','Large'],right=False)
pairs['congestion']=np.where(pairs['proxy_bbox_congestionado'].astype(bool),'Overlapping boxes','No overlap')
count=count[(count['split']=='test') & (count['escenario']=='Elegido en val')].copy()
assert len(count)==746 and count.groupby('modelo')['imagen_id'].nunique().eq(373).all()
assert count['fecha'].nunique()==5
print(f'✅ {len(pairs):,} pares emparejados y {len(count):,} filas de conteo; q33={q1:.5f}, q67={q2:.5f}')


## C. Construir matrices con números de observaciones

In [ ]:
MODELS=['YOLO11n-seg','Mask R-CNN']
DATES=sorted(ap['fecha'].unique())
APMAT=ap.pivot(index='fecha',columns='modelo',values='mAP50_95_mascaras').reindex(index=DATES,columns=MODELS)
APCOUNT=ap.pivot(index='fecha',columns='modelo',values='n_imagenes').reindex(index=DATES,columns=MODELS)
COUNTMAT=count.groupby(['fecha','modelo'])['error_abs'].mean().unstack('modelo').reindex(index=DATES,columns=MODELS)
COUNTN=count.groupby(['fecha','modelo']).size().unstack('modelo').reindex(index=DATES,columns=MODELS)
assert APMAT.notna().all().all() and COUNTMAT.notna().all().all()
IOUTABLE=(pairs.groupby(['modelo','tercil','congestion'],observed=True)
         .agg(mean_mask_IoU=('mask_iou','mean'),n_matched=('mask_iou','size'),
              mean_boundary_F1=('boundary_f1_2px','mean')).reset_index())
assert IOUTABLE['n_matched'].sum()==9069
EXCEL=MET/'CERDOS_19_Mapas_Calor.xlsx'
JPROTO=REP/'CERDOS_19_Mapas_Calor_Protocolo.json'
PROTO={'pipeline':'CERDOS_19_heatmaps_v1','sha256_fuentes':SOURCE_HASHES,
 'q33_validacion':q1,'q67_validacion':q2,
 'mAP':'COCOeval segm por fecha; valores del notebook 09',
 'IoU':'media SOLO sobre parejas GT-pred emparejadas (IoU>=0.5 y score>=0.25), notebook 11',
 'congestion':'indicador proxy: solapamiento IoU de cajas GT >=0.10',
 'conteo':'MAE por imagen sobre test, umbral elegido solo en validación, notebook 15',
 'advertencias':['No es evaluación nueva ni validación externa','Congestión no equivale a oclusión física',
  'Parejas emparejadas excluyen FN y no son una estimación global de calidad',
  'Fechas y prefijos de sesión no son conglomerados independientes demostrados']}
if JPROTO.exists() and json.loads(JPROTO.read_text(encoding='utf8'))['sha256_fuentes']!=SOURCE_HASHES:
    raise RuntimeError('Existe resultado 19 con distintas fuentes: revisar antes de sobrescribir')
with pd.ExcelWriter(EXCEL,engine='openpyxl') as w:
    ap.to_excel(w,sheet_name='AP_por_fecha',index=False)
    count.groupby(['fecha','modelo'],as_index=False).agg(MAE_conteo=('error_abs','mean'),n_imagenes=('error_abs','size')).to_excel(w,sheet_name='MAE_conteo_por_fecha',index=False)
    IOUTABLE.to_excel(w,sheet_name='IoU_tamano_congestion',index=False)
    pd.DataFrame([{'q33':q1,'q67':q2,'base':'GT validación; relativo al área imagen'}]).to_excel(w,sheet_name='Terciles_Val',index=False)
    pd.DataFrame([{'advertencia':a} for a in PROTO['advertencias']]).to_excel(w,sheet_name='Limitaciones',index=False)
JPROTO.write_text(json.dumps(PROTO,ensure_ascii=False,indent=2),encoding='utf8')
print('✅ Excel y protocolo guardados')


## D. Figuras científicas (PNG 600 dpi, PDF y SVG vectoriales)

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':12,
 'axes.labelsize':10,'figure.facecolor':'white','axes.facecolor':'white',
 'savefig.facecolor':'white','svg.fonttype':'none'})

def plot_heat(ax,vals,counts,rows,cols,title,fmt='.3f',cmap='YlGnBu',vmin=None,vmax=None):
    arr=np.asarray(vals,dtype=float)
    im=ax.imshow(arr,cmap=cmap,vmin=vmin,vmax=vmax,aspect='auto')
    ax.set_xticks(range(len(cols)),cols)
    ax.set_yticks(range(len(rows)),rows)
    ax.set_title(title,pad=12,weight='semibold')
    ax.tick_params(length=0)
    for i in range(arr.shape[0]):
      for j in range(arr.shape[1]):
        v=arr[i,j]; n=counts[i,j]
        if np.isfinite(v) and np.isfinite(n) and n>0:
          color='white' if im.norm(v)>0.68 else '#20242c'
          ax.text(j,i,f'{v:{fmt}}\n(n={int(n)})',ha='center',va='center',color=color,fontsize=9)
        else:
          ax.text(j,i,'n/a',ha='center',va='center',fontsize=9,color='#777777')
    for x in np.arange(-0.5,len(cols),1):ax.axvline(x,color='white',lw=1.2)
    for y in np.arange(-0.5,len(rows),1):ax.axhline(y,color='white',lw=1.2)
    return im

def save_figure(fig,stem):
    outs=[]
    for ext in ['png','pdf','svg']:
        path=FIG/(stem+'.'+ext)
        if ext=='png':fig.savefig(path,dpi=600,bbox_inches='tight')
        else:fig.savefig(path,bbox_inches='tight')
        outs.append(path)
    plt.show();plt.close(fig)
    return outs

fig,ax=plt.subplots(figsize=(7.2,5.0))
mat=APMAT.to_numpy(float);nmat=APCOUNT.to_numpy(float)
im=plot_heat(ax,mat,nmat,DATES,MODELS,'Mask AP50:95 by test capture date',fmt='.3f',vmin=0,vmax=1)
fig.colorbar(im,ax=ax,shrink=.85,label='COCO mask AP50:95')
fig.tight_layout()
files_out=save_figure(fig,'CERDOS_19_AP_Fecha_Modelo')

fig,ax=plt.subplots(figsize=(7.2,5.0))
mat=COUNTMAT.to_numpy(float);nmat=COUNTN.to_numpy(float)
im=plot_heat(ax,mat,nmat,DATES,MODELS,'Counting MAE by test capture date',fmt='.2f',cmap='YlOrRd',vmin=0)
fig.colorbar(im,ax=ax,shrink=.85,label='Mean absolute count error (pigs/image)')
fig.tight_layout()
files_out+=save_figure(fig,'CERDOS_19_MAE_Conteo_Fecha_Modelo')

fig,axs=plt.subplots(1,2,figsize=(12.5,4.7),sharey=True)
for ax,model in zip(axs,MODELS):
    tab=IOUTABLE[IOUTABLE['modelo']==model]
    dat=tab.pivot(index='tercil',columns='congestion',values='mean_mask_IoU').reindex(index=['Small','Medium','Large'],columns=['No overlap','Overlapping boxes'])
    num=tab.pivot(index='tercil',columns='congestion',values='n_matched').reindex(index=['Small','Medium','Large'],columns=['No overlap','Overlapping boxes'])
    im=plot_heat(ax,dat.to_numpy(float),num.to_numpy(float),dat.index.tolist(),dat.columns.tolist(),model+' | matched-mask IoU',fmt='.3f',vmin=.80,vmax=.98)
    ax.tick_params(axis='x',labelrotation=12)
fig.subplots_adjust(left=.08,right=.84,bottom=.16,top=.85,wspace=.20)
cax=fig.add_axes([.89,.25,.018,.53])
fig.colorbar(im,cax=cax,label='Mean mask IoU (matched pairs)')
files_out+=save_figure(fig,'CERDOS_19_IoU_Tamano_Congestion')
print('✅ Figuras exportadas:',len(files_out),'archivos')


## E. Empaquetar e identificar las limitaciones

In [ ]:
ZIP=EXP/'CERDOS_19_Mapas_Calor_Resultados.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in [EXCEL,JPROTO]+files_out:
        z.write(p,arcname=p.name)
with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None
print('✅ ZIP en Drive:',ZIP,'| Tamaño MiB:',round(ZIP.stat().st_size/1024**2,2))
files.download(str(ZIP))


**Siguiente:** revisar las figuras reales antes de incorporarlas al manuscrito; continuar con el Notebook 20 (GPU).